<a href="https://colab.research.google.com/github/Sara4h/boba/blob/main/Banking_Intent_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install datasets
from datasets import load_dataset

ds = load_dataset("PolyAI/banking77")
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}


In [7]:
names = ds["train"].features["label"].names

def to_example(ex):
    return {"messages": [
        {"role": "user", "content": ex["text"]},
        {"role": "assistant", "content": names[ex["label"]]},
    ]}

train = ds["train"].map(to_example)
print(train[0])

{'text': 'I am still waiting on my card?', 'label': 11, 'messages': [{'content': 'I am still waiting on my card?', 'role': 'user'}, {'content': 'card_arrival', 'role': 'assistant'}]}


In [6]:
!pip install unsloth

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",
    max_seq_length = 512,
    load_in_4bit = False,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    bias = "none", random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1125 07:49:40.196000 382 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1125 07:49:40.237000 382 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.11: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.11.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).


In [9]:
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

train_text = train.map(to_text)
print(train_text[0]["text"])

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

<|im_start|>user
I am still waiting on my card?<|im_end|>
<|im_start|>assistant
<think>

</think>

card_arrival<|im_end|>



In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_text,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 1,
        max_steps = 300,
        learning_rate = 2e-4,
        warmup_steps = 10,
        logging_steps = 20,
        optim = "adamw_8bit",
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/10003 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=2):   0%|          | 0/10003 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,003 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
20,3.464040
40,0.886498
60,0.471694
80,0.339781
100,0.214216
120,0.225871
140,0.138031
160,0.096211
180,0.129765
200,0.108930


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.


TrainOutput(global_step=300, training_loss=0.43296668291091917, metrics={'train_runtime': 205.9722, 'train_samples_per_second': 23.304, 'train_steps_per_second': 1.457, 'total_flos': 415222579200000.0, 'train_loss': 0.43296668291091917, 'epoch': 0.4792332268370607})

In [13]:
test = ds["test"].shuffle(seed=42).select(range(400))
print(test[0])

{'text': 'Are there restrictions for my disposable card since it does not seem to be working?', 'label': 72}


In [17]:
!pip install -q google-genai

In [29]:
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

correct = 0
for i in range(0, 100, 50):
    batch = test.select(range(i, i + 50))
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": t}],
            tokenize=False, add_generation_prompt=True)
        for t in batch["text"]
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=25, do_sample=False)
    answers = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    if i == 0:
        print([a for a in answers[:3]])
    for a, label in zip(answers, batch["label"]):
        clean = a.replace("<think>", "").replace("</think>", "").strip()
        if clean == names[label]:
            correct += 1

print("Qwen accuracy:", correct / 100)

Both `max_new_tokens` (=25) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['<think>\n\n</think>\n\ndisposable_card_not_working', '<think>\n\n</think>\n\npending_transfer', '<think>\n\n</think>\n\ncash_withdrawal_not_recognised']
Qwen accuracy: 0.86


In [30]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("Gemini_API_Key"))
label_list = "\n".join(names)
system = f"Classify the bank customer message into exactly one of these intents. Reply with only the intent name.\n\n{label_list}"

config = types.GenerateContentConfig(
    system_instruction=system,
    temperature=0,
    max_output_tokens=20,
)

In [26]:
try:
    r = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="hello",
        config=config,
    )
    print(r.text)
except Exception as e:
    print(str(e)[:600])

card_arrival


In [27]:
import time

MODEL = "gemini-3.1-flash-lite"
N = 100

gem_correct, in_tok, out_tok, failed = 0, 0, 0, 0
for text, label in zip(test["text"][:N], test["label"][:N]):
    r = None
    for attempt in range(3):
        try:
            r = client.models.generate_content(model=MODEL, contents=text, config=config)
            break
        except Exception as e:
            print("retrying:", str(e)[:100])
            time.sleep(20)
    if r is None:
        failed += 1
        continue
    if (r.text or "").strip() == names[label]:
        gem_correct += 1
    in_tok += r.usage_metadata.prompt_token_count
    out_tok += r.usage_metadata.candidates_token_count or 0
    time.sleep(4)

print("Gemini accuracy:", gem_correct / N, "| failed requests:", failed)
print("Input tokens:", in_tok, "Output tokens:", out_tok)

Gemini accuracy: 0.77 | failed requests: 0
Input tokens: 56468 Output tokens: 514


In [23]:
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemi

In [24]:
r = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=test["text"][0],
    config=config,
)
print(repr(r.text), "| true label:", names[test["label"][0]])


'disposable_card_limits' | true label: virtual_card_not_working
